In [54]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [55]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [56]:
df_produk = spark.createDataFrame(pd.DataFrame(data_target_cabang))
print("data_target_cabang:")
df_produk.show()
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi_t5))
print("data_transaksi_t5:")
df_transaksi.show(10)

data_target_cabang:
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+

data_transaksi_t5:
+--------+--------------------+----------+------------+------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|
+--------+--------------------+----------+------------+------------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|
|   TRX-5|Kesehatan & Kecan...|  Magelang|           4|  

In [58]:
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
ringkas_total = df_transaksi.groupBy("kota").agg(
    spark_sum(col("unit_terjual") * col("harga_satuan")).alias("total_pendapatan")
)

df_target = ringkas_total.join(df_produk, on="kota", how="inner")
hasil = df_target.withColumn("pendapatan_persen", (col("total_pendapatan") / col("target_bulanan") * 100 )
)

hasil.orderBy(col("pendapatan_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pendapatan_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [60]:
from pyspark.sql.window import Window

df_kategori = df_transaksi.withColumn("pendapatan_transaksi", col("unit_terjual") * col("harga_satuan"))\
    .groupBy("kota", "kategori")\
    .agg(spark_sum("pendapatan_transaksi").alias("total_pendapatan_transaksi")) 

window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan_transaksi").desc())

# Menambahkan kolom peringkat SETIAP transaksi di dalam kategorinya masing-masing
df_ranked = df_kategori.withColumn("peringkat", row_number().over(window_spec))

# Menampilkan hanya transaksi teratas (rank 1) di setiap kategori
df_ranked.filter(col("peringkat") == 1) \
    .orderBy("kota") \
    .show()

+----------+--------------------+--------------------------+---------+
|      kota|            kategori|total_pendapatan_transaksi|peringkat|
+----------+--------------------+--------------------------+---------+
|  Magelang|Kesehatan & Kecan...|                   7275000|        1|
| Purworejo|Kesehatan & Kecan...|                  10075000|        1|
|  Semarang|        Rumah Tangga|                  11125000|        1|
|      Solo|Kesehatan & Kecan...|                   8425000|        1|
|Yogyakarta|             Fashion|                  13325000|        1|
+----------+--------------------+--------------------------+---------+



In [61]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'")

hasil_sql = spark.sql('''
    SELECT t.kota, g.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target g ON t.kota = g.kota
    GROUP BY t.kota, g.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_sql.show()

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'


+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



### Analisis Kinerja Cabang

Berdasarkan hasil analisis pada bagian A dan B, cabang yang berkinerja paling baik adalah **Purworejo** dengan total penjualan sebesar **Rp45650000** dari target bulanan sebesar **Rp30000000**. Dengan demikian, pencapaian target cabang tersebut mencapai sekitar **152.16666666666669%**. Hasil ini menunjukkan bahwa cabang tersebut memiliki tingkat pencapaian penjualan yang paling tinggi dibandingkan cabang lainnya.

Sementara itu, cabang yang paling perlu mendapatkan perhatian manajemen adalah **Semarang**. Cabang tersebut hanya menghasilkan penjualan sebesar **Rp38175000** dari target **Rp55000000**, atau sekitar **69.4090909090909%** dari target. Selisih antara target dan realisasi mencapai **Rp16825000**. Berdasarkan angka tersebut, diperlukan perhatian lebih dari manajemen untuk mengevaluasi faktor yang menyebabkan pencapaian target masih rendah, seperti jumlah transaksi, kategori produk yang kurang diminati, atau tingkat penjualan di cabang tersebut.